# Residual decay

How the residual behaves after it is large - measured, and compared against the baseline that pure arithmetic already guarantees.

**This notebook contains no calculations.** Everything is computed by
`xauusd_quant.features` and `xauusd_quant.research` and read back from
`results/regression_research/<timeframe>/window_<N>/`. Generate those first:

```
xq regression-research --timeframe 1h --window 128
xq regression-research --timeframe 5m --all-windows
xq regression-research --all
```

No trading rule is defined or evaluated anywhere in this notebook. No
transaction costs are applied. Nothing here is a signal.

In [ ]:
import sys
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl

from xauusd_quant import load_config
from xauusd_quant.features import load_regression_config

config = load_config(ROOT / "config" / "data.yaml")
regression = load_regression_config(ROOT / "config" / "regression.yaml")

TIMEFRAME = "1h"          # change and re-run
WINDOW = 128              # change and re-run

print(f"instrument : {config.instrument}")
print(f"timezone   : {config.timezone.describe()}")
print(f"transform  : {regression.regression.price_transform} of "
      f"{regression.regression.price_source} {regression.regression.price_column}")
print(f"windows    : {list(regression.regression.windows)}")
print(f"fingerprint: {regression.fingerprint()}")

In [ ]:
import json

results = regression.results_dir(TIMEFRAME, WINDOW)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME} window={WINDOW}. Run:  "
        f"xq regression-research --timeframe {TIMEFRAME} --window {WINDOW}"
    )

def table(name):
    """Load a previously-generated regression table."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
provenance = summary["provenance"]
model = summary.get("model_summary", {})

print(f"generated  : {provenance['generated_utc']}")
print(f"window     : {provenance['regression_window']} bars on {provenance['timeframe']}")
print(f"valid fits : {provenance['observations']:,}")
print(f"span       : {provenance['requested_start']} -> {provenance['requested_end']}")
print(f"git        : {provenance['git']}")
for warning in summary.get("warnings", []):
    print(f"WARNING: {warning}")

## The decay coefficient

The regression is

```
delta eps_{t+1} = a + b * eps_t + eta_t
```

A negative `b` is consistent with large residuals being followed by movement
back toward zero. Two things travel with it, permanently:

1. **`b` is not an OU speed and not a half-life.** Prompt #3 places both
   outside scope, and neither is estimated here.
2. **Any bounded, zero-mean series gives `b < 0` mechanically.** The same
   regression is therefore fitted to a detrended random walk. The gap between
   the two is the informative quantity; `b` on its own is not.

In [ ]:
for key in ("decay_coefficient", "decay_coefficient_control", "decay_newey_west_t"):
    value = model.get(key)
    print(f"{key:<28} {value if value is None else f'{value:+.6f}'}")

b = model.get("decay_coefficient")
control = model.get("decay_coefficient_control")
if b is not None and control is not None:
    print(f"\ngap vs control             {b - control:+.6f}")
    print(f"ratio to control           {b / control:.3f}" if control else "")

## Conditional decay, by Z bin and horizon

Forward residuals are **outcomes**. The bin depends only on the Z-score at `t`;
nothing from after `t` enters the bin assignment. That is the whole discipline
of this table.

In [ ]:
table('decay_analysis')

### The probability that is easy to misread

`prob_move_toward_zero` is `P(|eps_{t+h}| < |eps_t|)`. **Its baseline is not
zero, and it is not 0.5 either once you condition on an extreme.**

For an independent standard-normal series with no predictability at all,
conditioning on `|Z| > 2` gives a value near **0.97**, purely because a large
draw is usually followed by a smaller one. A reading of 0.8 would therefore be
*worse* than noise, not evidence of mean reversion.

The number to look at is the gap between this table and the control, not the
level.

In [ ]:
decay = table('decay_analysis')
if decay is not None:
    pivot = decay.pivot(
        values="prob_move_toward_zero", index="z_bin", on="horizon",
        aggregate_function="first",
    )
    pivot

## Conditioning

Does the behaviour differ by trailing volatility, by the steepness of the local
trend, or by how well the window fitted a line? Each table is built the same
way, so they are directly comparable.

Quantile edges come from the whole sample. That is fine for describing history
and would be look-ahead in a live rule.

In [ ]:
table('volatility_conditioning')

In [ ]:
table('slope_conditioning')

In [ ]:
table('r_squared_conditioning')

## Stability over time

A relationship visible in every year is a different object from one carried by
a single period.

In [ ]:
table('yearly_stability')

In [ ]:
table('slope_by_year')

## What is still missing before any of this could matter

No transaction cost has been applied anywhere above. For reference, the
statistical research layer measured the typical reversal against the
round-trip spread and found it an order of magnitude short. Nothing in this
notebook changes that, and nothing here defines an entry, an exit or a size.